In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['subtype_for_TCGA-KIRC.tsv',
 'cases_for_TCGA-LUSC.tsv',
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'cases_for_TCGA-THYM.tsv',
 'cases_for_TCGA-PAAD.tsv',
 'cases_for_TCGA-LAML.tsv',
 'cases_for_TCGA-LIHC.tsv',
 'subtype_for_TCGA-KIRP.tsv',
 'subtype_for_TCGA-CHOL.tsv',
 'cases_for_TCGA-MESO.tsv',
 'cases_for_TCGA-BLCA.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_157ec9e0-ca6e-4da1-9233-f976cf0db433_sample_type_Primary_Tumor_stage_Stage_IV_fileid_a127fb1e-aa3b-4b6d-98b5-c141ffba9ae7.tsv',
 'cases_for_TCGA-KIRP.tsv',
 'cases_for_TCGA-LGG.tsv',
 'cases_for_TCGA-ESCA.tsv',
 'cases_for_TCGA-GBM.tsv',
 'subtype_for_TCGA-ACC.tsv',
 'cases_for_TCGA-TGCT.tsv',
 'subtype_for_TCGA-MESO.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_1783cac1-253a-40af-a9ac-48dfb20e1ab8_sample_type_Primary_Tumor_stage_Stage_IV_fileid_451ecdf8-433f-4d38-8ab6-3cc8d428ed95.tsv',
 'subtype_for_TCGA-OV.tsv',
 'cases_for_TCGA-UCEC.tsv',
 'cases_for_TCGA-HNSC.tsv',
 'subtype_for_T

### Methods

#### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

#### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

#### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

#### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

#### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

#### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(dfc))
dfc.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
dfc.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [9]:
i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

5 TCGA-BRCA Breast


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [10]:
force=False
verbose=True

i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))
df_subt

5 TCGA-BRCA Breast
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
df_cases 1098 df_subt 12


,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-BRCA,other,other,other,II,447
1,TCGA-BRCA,lobular,other,breast_lobular,missing,223
2,TCGA-BRCA,other,other,other,III,154
3,TCGA-BRCA,other,other,other,I,141
4,TCGA-BRCA,other,other,other,missing,85
5,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,missing,19
6,TCGA-BRCA,other,other,other,IV,16
7,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,missing,6
8,TCGA-BRCA,ductal,other,breast_ductal,II,2
9,TCGA-BRCA,clear_cell,other,clear_cell,II,2


In [11]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-BRCA,e3935ce4-64d3-4a66-ba11-d308b844b410,other,other,other,unknown
1,TCGA-BRCA,e3b555aa-7f0a-49c6-9b13-182c61a144c1,other,other,other,Stage IIIA
2,TCGA-BRCA,17ca61a2-607a-45ff-88fa-ef72e80bf891,other,other,other,Stage IA
3,TCGA-BRCA,7d681cc6-689d-41c8-9e84-e13733089ec9,other,other,other,Stage IIB
4,TCGA-BRCA,17f275c1-a0d4-487d-8f02-ea279584b4cd,other,other,other,Stage IA
5,TCGA-BRCA,189e1f27-7738-413a-a4d4-97d41d592a13,other,other,other,Stage IIA
6,TCGA-BRCA,95c53ecf-d8f1-4bcb-9b1a-c9a0542939f0,other,other,other,Stage IA
7,TCGA-BRCA,c0f88fc5-56b0-4255-bbd1-9a21ced8b37b,other,other,other,Stage IIIA
8,TCGA-BRCA,0f64edec-0f1f-4025-8a53-75f9534f7828,other,other,other,Stage IIA
9,TCGA-BRCA,c25898b4-eb33-42f4-bf3f-bf532a929e7d,lobular,other,breast_lobular,unknown


### Get all cases and their classifications

In [12]:

force=False
verbose=True

run_all = False

if run_all:

    for i in range(len(dfc)):
        print(i, end=' ')
        pid = dfc.iloc[i].pid
        primary_site = dfc.iloc[i].primary_site

        print(pid, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)



### Search for samples and files for each case given

- input: pid, subtype_global, tumor_class, subtype_tissue, and stage

In [13]:
force=False
verbose=True

i = 5
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)
df_subt

5 TCGA-BRCA Breast
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'


,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-BRCA,other,other,other,II,447
1,TCGA-BRCA,lobular,other,breast_lobular,missing,223
2,TCGA-BRCA,other,other,other,III,154
3,TCGA-BRCA,other,other,other,I,141
4,TCGA-BRCA,other,other,other,missing,85
5,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,missing,19
6,TCGA-BRCA,other,other,other,IV,16
7,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,missing,6
8,TCGA-BRCA,ductal,other,breast_ductal,II,2
9,TCGA-BRCA,clear_cell,other,clear_cell,II,2


In [14]:
i = 6
df_subt.iloc[i]

pid               TCGA-BRCA
subtype_global        other
tumor_class           other
subtype_tissue        other
sstage                   IV
n                        16
Name: 6, dtype: object

In [15]:
force=False
verbose=True

i = 6
row = df_subt.iloc[i]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
sstage = row.sstage

s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{sstage}'"
print(s_case)

df_sample = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             sstage=sstage, batch_size=200,
                                             force=force, verbose=verbose)

print(len(df_sample))

df_sample.head(3)


TCGA-BRCA subtype 'other' tumor 'other' subtype_tissue 'other' stage 'IV'
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
Table opened ((445, 11)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-BRCA_other_other_other_Stage_IV.tsv'
445


,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
0,32bc7cc4-d185-4594-bbee-4b96410be512,4ab5f84d-7ba2-49e2-862b-48ff8cff3c34,Primary Tumor,a2db9935-0427-49f1-97c0-41db6d012889,TCGA-A8-A08T-01A-11-A13A-20_RPPA_data.tsv,Protein Expression Quantification,TCGA-BRCA,other,other,other,Stage IV
1,32bc7cc4-d185-4594-bbee-4b96410be512,4ab5f84d-7ba2-49e2-862b-48ff8cff3c34,Primary Tumor,6b2fc2ac-a4b4-4e33-9ecc-8f3f98d4e33e,TCGA-BRCA.dfe66ee6-0535-42b6-b6b1-b961604d8a27...,Allele-specific Copy Number Segment,TCGA-BRCA,other,other,other,Stage IV
2,32bc7cc4-d185-4594-bbee-4b96410be512,4ab5f84d-7ba2-49e2-862b-48ff8cff3c34,Primary Tumor,125aede6-ea2e-4f2d-a8f1-31b9ec3464dd,TCGA_BRCA.ed76da0d-850e-4017-8bd4-f2e9e90d27f1...,Annotated Somatic Mutation,TCGA-BRCA,other,other,other,Stage IV


In [16]:
dfu = gdc.group_file_types(df_sample)
dfu

,data_type,n
0,Annotated Somatic Mutation,92
1,Raw Simple Somatic Mutation,64
2,Aligned Reads,42
3,Structural Rearrangement,32
4,Transcript Fusion,24
5,Gene Level Copy Number,24
6,Allele-specific Copy Number Segment,20
7,Slide Image,19
8,Copy Number Segment,18
9,Masked Intensities,12


In [17]:
lista = list(dfu.data_type)
lista.sort()

"; ".join(lista)

'Aggregated Somatic Mutation; Aligned Reads; Allele-specific Copy Number Segment; Annotated Somatic Mutation; Copy Number Segment; Gene Expression Quantification; Gene Level Copy Number; Intermediate Analysis Archive; Isoform Expression Quantification; Masked Copy Number Segment; Masked Intensities; Masked Somatic Mutation; Methylation Beta Value; Pathology Report; Protein Expression Quantification; Raw Intensities; Raw Simple Somatic Mutation; Simple Germline Variation; Slide Image; Splice Junction Quantification; Structural Rearrangement; Transcript Fusion; miRNA Expression Quantification'

### Get Any table (for tumor and control)

In [18]:
force=False
verbose=True

i=0
row = df_sample.iloc[i]

pid = row.pid
case_id = row.case_id
file_type = row.data_type
file_id = row.file_id
sample_type = row.sample_type
stage = row.stage


print(f"{pid}, {case_id}, {file_type}, {file_id}")

dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                 file_type=file_type, sample_type=sample_type, stage=stage, file_id=file_id, 
                                 force=force, verbose=verbose)
print(len(dft))
dft.head(6)

TCGA-BRCA, 32bc7cc4-d185-4594-bbee-4b96410be512, Protein Expression Quantification, a2db9935-0427-49f1-97c0-41db6d012889
Table opened ((487, 6)) at '/home/flavio/uv/perturb_agent/data/tcga/Protein_Expression_Quantification_for_TCGA-BRCA_case_32bc7cc4-d185-4594-bbee-4b96410be512_sample_type_Primary_Tumor_stage_Stage_IV_fileid_a2db9935-0427-49f1-97c0-41db6d012889.tsv'
487


,AGID,lab_id,catalog_number,set_id,peptide_target,protein_expression
0,AGID00100,882,sc-628,Old,1433BETA,-0.018320
1,AGID00111,913,sc-23957,Old,1433EPSILON,-0.036071
2,AGID00101,883,sc-1019,Old,1433ZETA,0.387080
3,AGID00001,2,9452,Old,4EBP1,0.016983
4,AGID00002,3,9456,Old,4EBP1_pS65,-0.604880
5,AGID00003,6,9459,Old,4EBP1_pT37T46,0.056122


### Get Expression table (for tumor and control)

#### 🧪 Mental model (important)

| Step	|  Needs strand? | Output | Column |
|---------|---------|---------|---------|
| Read assignment	|  YES |  counts | stranded_first |
| Normalization	| NO | TPM / FPKM | both: unstranded |
| Biological meaning | NO | gene expression| both: unstranded |

In [19]:
file_type = 'Gene Expression Quantification'

force=False
verbose=False

dft = pd.DataFrame()

df2 = df_sample[df_sample.data_type == file_type]

for i, row in df2.iterrows():
    pid = row.pid
    case_id = row.case_id
    file_type = row.data_type
    file_id = row.file_id
    sample_type = row.sample_type
    stage = row.stage

    print(f"{i}) {pid}, {case_id}, {file_type}, {file_id}")

    dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                     sample_type=sample_type, stage=stage, 
                                     file_type=file_type, file_id=file_id, 
                                     force=force, verbose=verbose)


dft.head(6)



8) TCGA-BRCA, 32bc7cc4-d185-4594-bbee-4b96410be512, Gene Expression Quantification, 8603da6b-3bb9-4fff-b409-c5af2afd53b3
129) TCGA-BRCA, 1783cac1-253a-40af-a9ac-48dfb20e1ab8, Gene Expression Quantification, 9985bb0b-d1cc-455b-80d9-f8007669cd5c
175) TCGA-BRCA, 1783cac1-253a-40af-a9ac-48dfb20e1ab8, Gene Expression Quantification, 451ecdf8-433f-4d38-8ab6-3cc8d428ed95
186) TCGA-BRCA, 157ec9e0-ca6e-4da1-9233-f976cf0db433, Gene Expression Quantification, a127fb1e-aa3b-4b6d-98b5-c141ffba9ae7
316) TCGA-BRCA, 128d198e-9b22-427c-90db-3714455f3a17, Gene Expression Quantification, 54600389-6de5-44ce-ada1-8a8508b50980
357) TCGA-BRCA, 01263518-5f7c-49dc-8d7e-84b0c03a6a63, Gene Expression Quantification, aa89d193-05a1-4f7f-9291-b9cf646656de


,gene_id,symbol,gene_type,unstranded,counts,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,911,446,465,12.8633,3.2599,3.1661
1,ENSG00000000005,TNMD,protein_coding,22,12,10,0.9547,0.2419,0.2350
2,ENSG00000000419,DPM1,protein_coding,4349,2189,2160,230.7761,58.4850,56.8023
3,ENSG00000000457,SCYL3,protein_coding,1420,1109,1155,13.2135,3.3487,3.2523
4,ENSG00000000460,C1orf112,protein_coding,1197,1075,1036,12.8419,3.2545,3.1608
5,ENSG00000000938,FGR,protein_coding,679,366,313,12.8589,3.2588,3.1650


In [20]:
np.unique(df_sample.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

In [21]:
df_sample.columns

Index(['case_id', 'sample_id', 'sample_type', 'file_id', 'file_name',
       'data_type', 'pid', 'subtype_global', 'tumor_class', 'subtype_tissue',
       'stage'],
      dtype='str')

### Get Normal and Tumor datasets given case_id and data_type

In [22]:
case_id = '1783cac1-253a-40af-a9ac-48dfb20e1ab8'
data_type = 'Gene Expression Quantification'

df_normal, df_tumor = gdc.get_tumor_normal_tables(df_sample=df_sample, case_id=case_id, data_type=data_type)

len(df_normal), len(df_tumor)

(1, 5)

In [23]:
df_normal

,file_id,data_type,sample_type
0,9985bb0b-d1cc-455b-80d9-f8007669cd5c,Gene Expression Quantification,Solid Tissue Normal


In [24]:
df_tumor

,file_id,data_type,sample_type
0,8603da6b-3bb9-4fff-b409-c5af2afd53b3,Gene Expression Quantification,Primary Tumor
1,451ecdf8-433f-4d38-8ab6-3cc8d428ed95,Gene Expression Quantification,Primary Tumor
2,a127fb1e-aa3b-4b6d-98b5-c141ffba9ae7,Gene Expression Quantification,Primary Tumor
3,54600389-6de5-44ce-ada1-8a8508b50980,Gene Expression Quantification,Primary Tumor
4,aa89d193-05a1-4f7f-9291-b9cf646656de,Gene Expression Quantification,Primary Tumor


### Merge Expression Tables for DEGs' calculation

In [25]:
dfa_normal, dfa_tumor = gdc.merge_normal_tumor_tables(pid=pid, df_normal=df_normal, df_tumor=df_tumor)

dfa_normal.shape, dfa_tumor.shape

((60660, 4), (61980, 8))

In [26]:
dfa_normal.head(3)

,gene_id,symbol,gene_type,counts_1
0,ENSG00000000003,TSPAN6,protein_coding,1511
1,ENSG00000000005,TNMD,protein_coding,167
2,ENSG00000000419,DPM1,protein_coding,948


In [27]:
dfa_tumor.head(3)

,gene_id,symbol,gene_type,counts_1,counts_2,counts_3,counts_4,counts_5
0,ENSG00000000003,TSPAN6,protein_coding,1714,2222,910,1335,446
1,ENSG00000000005,TNMD,protein_coding,3,32,6,0,12
2,ENSG00000000419,DPM1,protein_coding,847,1045,917,1584,2189


In [28]:
i=100
dfa_tumor.iloc[i:i+30]

,gene_id,symbol,gene_type,counts_1,counts_2,counts_3,counts_4,counts_5
100,ENSG00000004948,CALCR,protein_coding,17,26,27,2,22
101,ENSG00000004961,HCCS,protein_coding,567,651,878,1188,1667
102,ENSG00000004975,DVL2,protein_coding,1354,1126,1897,675,1140
103,ENSG00000005001,PRSS22,protein_coding,199,242,399,1512,249
104,ENSG00000005007,UPF1,protein_coding,4541,4820,4021,2262,5298
105,ENSG00000005020,SKAP2,protein_coding,464,2499,1797,1068,992
106,ENSG00000005022,SLC25A5,protein_coding,5092,5346,9037,4825,9933
107,ENSG00000005059,MCUB,protein_coding,1052,510,343,342,307
108,ENSG00000005073,HOXA11,protein_coding,38,30,15,4,32
109,ENSG00000005075,POLR2J,protein_coding,602,675,2673,1319,1136


### Calc DEGs

In [29]:
cdegs = CALC_DEGS(root_data=root_data)

cdegs.libs_dir, cdegs.root_data

(PosixPath('/home/flavio/uv/perturb_agent/src/libs'),
 PosixPath('/home/flavio/uv/perturb_agent/data/tcga'))

In [30]:
dfa_normal2 = cdegs.deduplicate_by_max_reads(dfa_normal)
len(dfa_normal2), len(dfa_normal)

(60616, 60660)

In [31]:
dfa_tumor2 = cdegs.deduplicate_by_max_reads(dfa_tumor)
len(dfa_tumor2), len(dfa_tumor)

(60616, 61980)

In [32]:
df_lfc = cdegs.run_deg_rscript(df_tumor=dfa_tumor2, df_normal=dfa_normal2,
                                method="edger",  manual_dispersion=0.1, min_total_count=10, 
                                merge_how="inner", keep_temp=False)
print(len(df_lfc))

25250


In [35]:
df_lfc = df_lfc.rename(columns={"log2FoldChange": "lfc", "padj": "fdr"})
df_lfc.head(3)

,gene_id,symbol,gene_type,lfc,logCPM,statistic,pvalue,fdr,method
0,ENSG00000137440,FGFBP1,protein_coding,-9.185835,2.257341,NaN,9.425175e-67,1.704767e-62,edgeR_exact_manualDisp_0.1
1,ENSG00000244734,HBB,protein_coding,-7.403060,6.420277,NaN,1.350310e-66,1.704767e-62,edgeR_exact_manualDisp_0.1
2,ENSG00000188536,HBA2,protein_coding,-7.298696,3.683474,NaN,1.113013e-60,9.367861e-57,edgeR_exact_manualDisp_0.1


In [36]:
lfc_cutoff = 1.0
fdr_cutoff = .05

df_degs = df_lfc[ (df_lfc.lfc >= lfc_cutoff) & (df_lfc.fdr < fdr_cutoff)].copy()
print(len(df_degs))

df_degs.head(30)

2365


,gene_id,symbol,gene_type,lfc,logCPM,statistic,pvalue,fdr,method
51,ENSG00000141668,CBLN2,protein_coding,12.118940,7.425111,NaN,1.871379e-26,9.086985e-24,edgeR_exact_manualDisp_0.1
85,ENSG00000147381,MAGEA4,protein_coding,14.202259,5.905617,NaN,1.119542e-22,3.287028e-20,edgeR_exact_manualDisp_0.1
96,ENSG00000123500,COL10A1,protein_coding,8.568603,7.461871,NaN,4.308427e-21,1.121523e-18,edgeR_exact_manualDisp_0.1
104,ENSG00000170419,VSTM2A,protein_coding,13.547065,5.254319,NaN,1.024209e-20,2.462980e-18,edgeR_exact_manualDisp_0.1
109,ENSG00000105141,CASP14,protein_coding,9.514498,5.759617,NaN,2.758795e-20,6.332688e-18,edgeR_exact_manualDisp_0.1
114,ENSG00000105388,CEACAM5,protein_coding,9.018059,5.831233,NaN,5.199101e-20,1.141542e-17,edgeR_exact_manualDisp_0.1
116,ENSG00000105048,TNNT1,protein_coding,8.719954,5.935822,NaN,6.613162e-20,1.427199e-17,edgeR_exact_manualDisp_0.1
117,ENSG00000060718,COL11A1,protein_coding,8.084070,7.275420,NaN,7.168558e-20,1.533950e-17,edgeR_exact_manualDisp_0.1
135,ENSG00000170373,CST1,protein_coding,12.908342,4.618917,NaN,8.208498e-19,1.515591e-16,edgeR_exact_manualDisp_0.1
139,ENSG00000117148,ACTL8,protein_coding,12.845358,4.548618,NaN,1.245940e-18,2.247143e-16,edgeR_exact_manualDisp_0.1


In [ ]:
fname = 'degs.txt'

text = "\n".join(df_degs.gene_id)

write_txt(text, fname, root_data)

'ENSG00000141668, ENSG00000147381, ENSG00000123500, ENSG00000170419, ENSG00000105141, ENSG00000105388, ENSG00000105048, ENSG00000060718, ENSG00000170373, ENSG00000117148, ENSG00000122584, ENSG00000253802, ENSG00000221867, ENSG00000235123, ENSG00000234722, ENSG00000174469, ENSG00000204832, ENSG00000069011, ENSG00000130294, ENSG00000143556, ENSG00000211892, ENSG00000189001, ENSG00000223808, ENSG00000185686, ENSG00000084628, ENSG00000253877, ENSG00000137745, ENSG00000197172, ENSG00000075388, ENSG00000203972, ENSG00000158639, ENSG00000198033, ENSG00000159184, ENSG00000156096, ENSG00000230838, ENSG00000107159, ENSG00000019505, ENSG00000104321, ENSG00000162482, ENSG00000105664, ENSG00000263639, ENSG00000204539, ENSG00000211452, ENSG00000046774, ENSG00000078898, ENSG00000166961, ENSG00000176566, ENSG00000249203, ENSG00000118785, ENSG00000196611, ENSG00000009709, ENSG00000070985, ENSG00000166509, ENSG00000123407, ENSG00000260318, ENSG00000135346, ENSG00000178372, ENSG00000256612, ENSG000001249